# SQLAlchemy (2.0 style)

We'll start off using the world.sqlite database again.

This notebook uses modern SQLAlchemy 2.x Core patterns.

In [ ]:
#!/usr/bin/env python
from sqlalchemy import create_engine, MetaData, Table
from sqlalchemy import select, desc

# Create an Engine object which is our handle into the database.
engine = create_engine('sqlite:///../world.sqlite')

# Read metadata from the existing database.
DBInfo = MetaData()

In [ ]:
# Auto-create the city object based on reflected metadata.
city = Table('city', DBInfo, autoload_with=engine)

# Mirroring: SELECT Id, Name, Population FROM city LIMIT 10;
query = select(city.c.ID, city.c.Name, city.c.Population).limit(10)
with engine.connect() as conn:
    result = conn.execute(query)
    for row in result:
        print(row)

In [ ]:
# Mirroring: SELECT Id, Name, Population FROM city WHERE Population > 5000000;
query = select(city.c.ID, city.c.Name, city.c.Population).where(city.c.Population > 5000000)
with engine.connect() as conn:
    result = conn.execute(query)
    for row in result:
        print(row)

In [ ]:
# Mirroring:
# SELECT Id, Name, Population FROM city
# WHERE Population > 5000000 ORDER BY Population DESC LIMIT 10;
query = (
    select(city.c.ID, city.c.Name, city.c.Population)
    .where(city.c.Population > 5000000)
    .order_by(desc(city.c.Population))
    .limit(10)
)
with engine.connect() as conn:
    result = conn.execute(query)
    for row in result:
        print(row)

In [ ]:
# Mirroring:
# SELECT Name, Language, Percentage
# FROM country JOIN countrylanguage ON Code = CountryCode
# WHERE Language='Portuguese' ORDER BY Percentage DESC;
country = Table('country', DBInfo, autoload_with=engine)
countrylanguage = Table('countrylanguage', DBInfo, autoload_with=engine)

query = (
    select(country.c.Name, countrylanguage.c.Language, countrylanguage.c.Percentage)
    .select_from(country.join(countrylanguage, country.c.Code == countrylanguage.c.CountryCode))
    .where(countrylanguage.c.Language == 'Portuguese')
    .order_by(desc(countrylanguage.c.Percentage))
)
print(query)

with engine.connect() as conn:
    result = conn.execute(query)
    for row in result:
        print(row)

## Inserting

This section uses SQLAlchemy 2.x insert patterns with explicit transaction handling.

In [ ]:
print(city.insert())

In [ ]:
my_insert = city.insert().values(
    Name='Gainesville',
    CountryCode='USA',
    District='TX',
    Population=16612
)
print(my_insert)

In [ ]:
with engine.begin() as conn:
    conn.execute(my_insert)

In [ ]:
query = select(city.c.ID, city.c.Name, city.c.District, city.c.Population).where(city.c.Name.like('Gain%'))
with engine.connect() as conn:
    result = conn.execute(query)
    for row in result:
        print(row)

## Creating a database in SQLAlchemy 2.x


In [ ]:
from sqlalchemy import create_engine

engine_1 = create_engine('sqlite:///test.sqlite', echo=True)

In [ ]:
from sqlalchemy import Table, Column, Integer, String, MetaData, ForeignKey

metadata = MetaData()

users = Table(
    'users', metadata,
    Column('id', Integer, primary_key=True),
    Column('name', String),
    Column('fullname', String),
)

addresses = Table(
    'addresses', metadata,
    Column('id', Integer, primary_key=True),
    Column('user_id', Integer, ForeignKey('users.id')),
    Column('email_address', String, nullable=False),
)

In [ ]:
metadata.create_all(engine_1)